In [1]:
pip install -U transformers accelerate peft datasets soundfile librosa

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torchaudio
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM, WhisperProcessor, WhisperModel


class AudioPrefixQwen(nn.Module):
    """
    Whisper encoder -> pool+proj -> prefix embeddings -> Qwen2 CausalLM
    (device + dtype safe with device_map="auto")
    """
    def __init__(
        self,
        whisper_id="openai/whisper-base",
        qwen_id="Qwen/Qwen2-1.5B-Instruct",
        n_audio_tokens=32,
        whisper_fp32=True,   # safer; keep Whisper in fp32, cast only prefix to qwen dtype
    ):
        super().__init__()

        # 1) Whisper encoder
        self.whisper = WhisperModel.from_pretrained(whisper_id)
        self.processor = WhisperProcessor.from_pretrained(whisper_id)

        # 2) Qwen2 LLM (sharded if needed)
        self.qwen = AutoModelForCausalLM.from_pretrained(
            qwen_id,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(qwen_id, use_fast=True)

        # Qwen often has no pad token; set to eos for safety
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        whisper_dim = self.whisper.config.d_model
        qwen_dim = self.qwen.config.hidden_size
        self.n_audio_tokens = n_audio_tokens

        self.pool = nn.AdaptiveAvgPool1d(self.n_audio_tokens)
        self.proj = nn.Linear(whisper_dim, qwen_dim)

        # Truth: use Qwen input embedding device/dtype
        emb = self.qwen.get_input_embeddings().weight
        self.qwen_in_device = emb.device
        self.qwen_in_dtype = emb.dtype

        # Move modules to the correct device
        self.pool.to(self.qwen_in_device)
        self.proj.to(self.qwen_in_device)

        # Whisper: keep fp32 by default (more stable), but move to same device
        if whisper_fp32:
            self.whisper.to(self.qwen_in_device)  # keep fp32
        else:
            self.whisper.to(self.qwen_in_device, dtype=self.qwen_in_dtype)

        # Freeze Whisper (optional)
        for p in self.whisper.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def encode_audio(self, audio_1d, sampling_rate=16000):
        """
        audio_1d: 1D torch tensor or numpy array (mono)
        returns: (1, n_audio_tokens, qwen_dim) on Qwen device/dtype
        """
        if isinstance(audio_1d, torch.Tensor):
            audio_np = audio_1d.detach().cpu().numpy()
        else:
            audio_np = audio_1d

        feat = self.processor(audio_np, sampling_rate=sampling_rate, return_tensors="pt")
        input_features = feat.input_features.to(self.qwen_in_device)

        enc = self.whisper.encoder(input_features=input_features).last_hidden_state
        # (1, T, whisper_dim)

        x = enc.transpose(1, 2)      # (1, whisper_dim, T)
        x = self.pool(x)             # (1, whisper_dim, n_audio_tokens)
        x = x.transpose(1, 2)        # (1, n_audio_tokens, whisper_dim)

        # If whisper is fp32 and qwen is fp16, proj weights might be fp32; cast before proj
        x = x.to(self.qwen_in_device, dtype=self.proj.weight.dtype)
        x = self.proj(x)             # (1, n_audio_tokens, qwen_dim)

        # CRITICAL: match Qwen dtype/device
        x = x.to(device=self.qwen_in_device, dtype=self.qwen_in_dtype)
        return x

    def forward(self, audio_prefix_embeds, input_ids, attention_mask=None, labels=None):
        device = self.qwen_in_device
        dtype = self.qwen_in_dtype

        audio_prefix_embeds = audio_prefix_embeds.to(device=device, dtype=dtype)
        input_ids = input_ids.to(device)

        tok_embeds = self.qwen.get_input_embeddings()(input_ids)  # (B,L,qwen_dim), already dtype=dtype
        inputs_embeds = torch.cat([audio_prefix_embeds, tok_embeds], dim=1)

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids, device=device)
        else:
            attention_mask = attention_mask.to(device)

        prefix_mask = torch.ones(
            (input_ids.size(0), self.n_audio_tokens),
            device=device,
            dtype=attention_mask.dtype
        )
        attn = torch.cat([prefix_mask, attention_mask], dim=1)

        if labels is not None:
            labels = labels.to(device)
            prefix_labels = torch.full(
                (labels.size(0), self.n_audio_tokens),
                -100,
                device=device,
                dtype=labels.dtype
            )
            full_labels = torch.cat([prefix_labels, labels], dim=1)
        else:
            full_labels = None

        return self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attn,
            labels=full_labels
        )

    @torch.no_grad()
    def generate(self, audio_prefix_embeds, prompt_text, max_new_tokens=64):
        device = self.qwen_in_device
        dtype = self.qwen_in_dtype

        audio_prefix_embeds = audio_prefix_embeds.to(device=device, dtype=dtype)

        inputs = self.tokenizer(prompt_text, return_tensors="pt", padding=True).to(device)
        tok_embeds = self.qwen.get_input_embeddings()(inputs.input_ids)
        inputs_embeds = torch.cat([audio_prefix_embeds, tok_embeds], dim=1)

        prefix_mask = torch.ones(
            (inputs.input_ids.size(0), self.n_audio_tokens),
            device=device,
            dtype=inputs.attention_mask.dtype
        )
        attn = torch.cat([prefix_mask, inputs.attention_mask], dim=1)

        out = self.qwen.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
        return self.tokenizer.decode(out[0], skip_special_tokens=True)


2026-01-13 20:30:23.310095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768336223.341822    1003 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768336223.351214    1003 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768336223.370518    1003 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768336223.370542    1003 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768336223.370545    1003 computation_placer.cc:177] computation placer alr

In [3]:
# Loading the model and seeing the summary

model = AudioPrefixQwen(
        whisper_id="openai/whisper-base",
        qwen_id="Qwen/Qwen2-1.5B-Instruct",
        n_audio_tokens=32
    )

print(model)

`torch_dtype` is deprecated! Use `dtype` instead!


AudioPrefixQwen(
  (whisper): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm)

In [ ]:
# Correct Training Loop without LORA in Qwen
import torch
import torch.optim as optim
import random

device = model.qwen_in_device
model.train()

optimizer = optim.AdamW(model.proj.parameters(), lr=1e-4)

num_samples = 5
sr = 16000
audio_len = sr * 3

audios, targets = [], []
for _ in range(num_samples):
    audio = torch.empty(audio_len).uniform_(-0.5, 0.5)
    audio = audio / (audio.abs().max() + 1e-6)
    audios.append(audio)
    targets.append(random.choice(["real", "fake"]))

prompt = "Is this audio real or fake?"
prompt_prefix = prompt + " "

epochs = 10

for epoch in range(epochs):
    total_loss = 0.0
    steps = 0

    for audio, target in zip(audios, targets):
        audio_prefix = model.encode_audio(audio)

        # Tokenize EXACTLY
        prefix_ids = model.tokenizer(
            prompt_prefix, add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        ans_ids = model.tokenizer(
            target, add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        input_ids = torch.cat([prefix_ids, ans_ids], dim=1)
        attention_mask = torch.ones_like(input_ids)

        labels = torch.cat(
            [torch.full_like(prefix_ids, -100), ans_ids], dim=1
        )

        out = model(
            audio_prefix_embeds=audio_prefix,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = out.loss
        assert torch.isfinite(loss), "Loss became NaN"

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.proj.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {total_loss/steps:.4f}")


In [10]:
# Correct Training Loop wit LORA
! pip -q install peft

import torch
from peft import LoraConfig, get_peft_model, TaskType

device = model.qwen_in_device

# -----------------------------
# Freeze Whisper encoder (audio side frozen)
# -----------------------------
for p in model.whisper.parameters():
    p.requires_grad = False

# -----------------------------
# Ensure projection is trainable (and stable in fp32)
# -----------------------------
for p in model.proj.parameters():
    p.requires_grad = True
model.proj = model.proj.to(device=device, dtype=torch.float32)

# -----------------------------
# Add LoRA to Qwen (base frozen, adapters trainable)
# -----------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model.qwen = get_peft_model(model.qwen, lora_cfg)
model.qwen.print_trainable_parameters()

# Put model in train mode
model.train()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


AudioPrefixQwen(
  (whisper): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 512)
      (layers): ModuleList(
        (0-5): 6 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=False)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm)

In [14]:
# Training with LORA very Important and base code for all the ALM works
import torch
import torch.optim as optim
import random

device = model.qwen_in_device
model.train()

# Collect ONLY trainable parameters: proj + LoRA
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=2e-4)

num_samples = 5
sr = 16000
audio_len = sr * 3

audios, targets = [], []
for _ in range(num_samples):
    audio = torch.empty(audio_len).uniform_(-0.5, 0.5)
    audio = audio / (audio.abs().max() + 1e-6)   # keep audio valid for Whisper
    audios.append(audio)
    targets.append(random.choice(["real", "fake"]))

prompt = "Is this audio real or fake?"
prompt_prefix = prompt + " "

epochs = 10

for epoch in range(epochs):
    total_loss = 0.0
    steps = 0

    for audio, target in zip(audios, targets):
        # audio encoder frozen => encode_audio() no_grad is OK
        audio_prefix = model.encode_audio(audio)

        prefix_ids = model.tokenizer(
            prompt_prefix, add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        ans_ids = model.tokenizer(
            target, add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        input_ids = torch.cat([prefix_ids, ans_ids], dim=1)
        attention_mask = torch.ones_like(input_ids)

        labels = torch.cat([torch.full_like(prefix_ids, -100), ans_ids], dim=1)

        out = model(
            audio_prefix_embeds=audio_prefix,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = out.loss
        assert torch.isfinite(loss), "Loss became NaN/Inf"

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {total_loss/steps:.4f}")


AssertionError: Loss became NaN/Inf

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters:     {total_params / 1e6:.2f} M")
print(f"Trainable Parameters: {trainable_params / 1e6:.2f} M")
print(f"Trainable Percentage: {100 * trainable_params / total_params:.4f}%")

In [13]:
# Demo Inference Loop

model.eval()

print("\nInference Results:")
print("-" * 50)

with torch.no_grad():
    for i, audio in enumerate(audios):
        audio_prefix = model.encode_audio(audio)

        # generation: only give prompt (not the answer)
        pred = model.generate(
            audio_prefix_embeds=audio_prefix,
            prompt_text=prompt,
            max_new_tokens=3
        )

        print(f"Sample {i+1}")
        print(f"GT  : {targets[i]}")
        print(f"Gen : {pred}")
        print("-" * 50)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Inference Results:
--------------------------------------------------
Sample 1
GT  : real
Gen : !!!
--------------------------------------------------
Sample 2
GT  : fake
Gen : !!!
--------------------------------------------------
Sample 3
GT  : real
Gen : !!!
--------------------------------------------------
Sample 4
GT  : fake
Gen : !!!
--------------------------------------------------
Sample 5
GT  : real
Gen : !!!
--------------------------------------------------


In [ ]:
# Loading the model and seeing with a random input and output just for seeing

if __name__ == "__main__":


    url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
    path = "sample.flac"

    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(path, "wb") as f:
       f.write(r.content)

    wav, sr = torchaudio.load(path)
    if wav.size(0) > 1:
       wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
       wav = torchaudio.functional.resample(wav, sr, 16000)

    audio = wav.squeeze(0)


    model = AudioPrefixQwen(
        whisper_id="openai/whisper-base",
        qwen_id="Qwen/Qwen2-1.5B-Instruct",
        n_audio_tokens=32
    )

    prefix = model.encode_audio(audio, sampling_rate=sr)

    # NOTE: Without training, output will be random / generic.
    prompt = "Listen to the audio and guess what word is spoken. Reply with one word Yes or No"
    print(model.generate(prefix, prompt_text=prompt, max_new_tokens=32))